# Introduction to LangChain

## Initial setup

### Set API key for Groq
Click [here](https://console.groq.com/keys) to create API key for Groq, if not already created.

In [2]:
import os, json, re, getpass
from dotenv import load_dotenv

load_dotenv( override=True)

True

In [3]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

In [4]:
# if "TEST_API_KEY" not in os.environ:
#     os.environ["TEST_API_KEY"] = getpass.getpass("TEST API Key: ")

In [5]:
if os.environ["GROQ_API_KEY"]:
    print(f"Groq API Key exists and begins {os.environ["GROQ_API_KEY"][:4]}")
else:
    print("Groq API Key not set (and this is optional)")

Groq API Key exists and begins gsk_


## LangChain Components

### LLM / ChatModel

**Note** on **init_chat_model**: init_chat_model is just a helper method and under the hood, it will still be calling the specific Chat models only (like ChatOpenAI etc.). The only benefit of using init_chat_model is that the initialization is standard across providers, which is useful. 

See the source code of this method here for better details. Note that line 79 has init_chat_model() function and if model and model_provider are specified (which we do in class), then it returns an instance of _ConfigurableModel class (line 332). Then this class definition (line 554), returns the model in line 610 using _init_chat_model_helper() method. This method definition (line 339) actually returns ChatOllama() model instance (line 400). So it's the same thing.


In [ ]:
#Using LangChain
from langchain.chat_models import init_chat_model

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq")

In [18]:
llm_response = llm.invoke("what is ECG?")
llm_response

AIMessage(content='**ECG (Electrocardiogram)**  \n\n| Aspect | Details |\n|--------|----------|\n| **Definition** | A non‑invasive test that records the electrical activity of the heart over a short period (usually a few seconds to a few minutes). |\n| **What it measures** | The depolarization (activation) and repolarization (recovery) of the atria and ventricles, displayed as voltage‑versus‑time waveforms. |\n| **How it works** | Surface electrodes are placed on the skin (usually 10\u202f–\u202f12 standard positions). Each electrode detects tiny voltage changes (≈1\u202fmV) generated by the cardiac muscle and transmits them to a recorder or digital system, which plots the signals as the familiar “P‑QRS‑T” complexes. |\n| **Standard lead systems** | • **12‑lead ECG** – 3 limb leads (I, II, III), 3 augmented limb leads (aVR, aVL, aVF), and 6 precordial (chest) leads (V1‑V6). <br>• **Holter / ambulatory ECG** – continuous recording (24‑48\u202fh or longer). <br>• **Event monitors, patch 

In [20]:
print("type of response", type(llm_response))

type of response <class 'langchain_core.messages.ai.AIMessage'>


In [21]:
# print(llm_response.content)
# display(llm_response.response_metadata)
display(llm_response.usage_metadata)

{'input_tokens': 75, 'output_tokens': 1337, 'total_tokens': 1412}

In [22]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\
                  Ensure proper sentence structure, clarity, and readability.\
                  Retain the core message of the original text while making the necessary corrections"),
    HumanMessage(content="hey can you send me that report by tomorrow thx"),
]

ai_response = llm.invoke(messages)
print(ai_response.content)

Hey, can you send me that report by tomorrow? Thanks.


In [23]:
ai_response

AIMessage(content='Hey, can you send me that report by tomorrow? Thanks.', additional_kwargs={'reasoning_content': 'The user wants us to correct spelling, grammar, punctuation, sentence structure, clarity, readability, retain core message. The original text: "hey can you send me that report by tomorrow thx". We need to produce corrected version. Probably: "Hey, can you send me that report by tomorrow? Thanks." Ensure proper capitalization, punctuation. That\'s it.\n\nWe need to output corrected text. Probably just the corrected version.'}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 127, 'total_tokens': 235, 'completion_time': 0.227912575, 'prompt_time': 0.029861672, 'queue_time': 0.443219543, 'total_time': 0.257774247, 'completion_tokens_details': {'reasoning_tokens': 86}}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--8fd6459b-8ab6-4a38-9e7

In [24]:
# followup conversation
messages.append(ai_response)

In [25]:
#Ask a follow-up question
messages.append(HumanMessage(content="can you make the tone a bit informal"))

In [26]:
ai_response = llm.invoke(messages)
# print(ai_response)
print(ai_response.content)

Hey, could you send me that report by tomorrow? Thanks!


#### Doing without LangChain

In [27]:
#Without LangChain - how would we initialize our LLM?
from openai import OpenAI

model_name = "openai/gpt-oss-120b"
llm_api = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")

In [28]:
# #Below is LangChain's messages format
# messages = [

#     SystemMessage(content="Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\
#                   Ensure proper sentence structure, clarity, and readability.\
#                   Retain the core message of the original text while making the necessary corrections"),
#     HumanMessage(content="hey can you send me that report by tomorrow thx"),
# ]

#Below is messages in OpenAI format
messages_openai = [
    {'role':"system", 'content':"Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\
     Ensure proper sentence structure, clarity, and readability.\
     Retain the core message of the original text while making the necessary corrections"},
     
     {'role':"user", 'content':"hey can you send me that report by tomorrow thx"}
]


In [29]:
ai_response_openai = llm_api.chat.completions.create(model= model_name,
                                messages=messages_openai)

In [30]:
ai_response_openai

ChatCompletion(id='chatcmpl-1db4209a-c3ba-4120-b2d3-ef87a045967a', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hey, can you send me that report by tomorrow? Thanks.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='User: "hey can you ...". They ask to correct text. Provide corrected version. Should be polite. Return corrected text.'))], created=1788802276, model='openai/gpt-oss-120b', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_60e4b492db', usage=CompletionUsage(completion_tokens=49, prompt_tokens=127, total_tokens=176, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=27, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.353046349, prompt_time=0.005642338, completion_time=0.103162846, total_time=0.108805184), usage_breakdown=None, x_groq={'id': 

In [31]:
ai_response_openai_formatted = ai_response_openai.choices[0].message.content
print(ai_response_openai_formatted)

Hey, can you send me that report by tomorrow? Thanks.


In [32]:
messages_openai

[{'role': 'system',
  'content': 'Detect and correct all spelling, grammatical, and punctuation errors in the provided text.     Ensure proper sentence structure, clarity, and readability.     Retain the core message of the original text while making the necessary corrections'},
 {'role': 'user',
  'content': 'hey can you send me that report by tomorrow thx'}]

In [33]:
#Append the AI message
messages_openai.append(
    {'role': "assistant",
     'content': ai_response_openai_formatted}
)

In [34]:
messages_openai

[{'role': 'system',
  'content': 'Detect and correct all spelling, grammatical, and punctuation errors in the provided text.     Ensure proper sentence structure, clarity, and readability.     Retain the core message of the original text while making the necessary corrections'},
 {'role': 'user',
  'content': 'hey can you send me that report by tomorrow thx'},
 {'role': 'assistant',
  'content': 'Hey, can you send me that report by tomorrow? Thanks.'}]

In [35]:
#Ask a follow-up question
#LangChain version below
# messages.append(HumanMessage(content="can you make the tone a bit informal"))

#OpenAI version below
messages_openai.append(
    {'role':"user",
     'content':"can you make the tone a bit informal"}
)

In [36]:
messages_openai

[{'role': 'system',
  'content': 'Detect and correct all spelling, grammatical, and punctuation errors in the provided text.     Ensure proper sentence structure, clarity, and readability.     Retain the core message of the original text while making the necessary corrections'},
 {'role': 'user',
  'content': 'hey can you send me that report by tomorrow thx'},
 {'role': 'assistant',
  'content': 'Hey, can you send me that report by tomorrow? Thanks.'},
 {'role': 'user', 'content': 'can you make the tone a bit informal'}]

In [37]:
ai_response_openai = llm_api.chat.completions.create(
    model=model_name,
    messages=messages_openai
)

In [38]:
type(ai_response_openai)

openai.types.chat.chat_completion.ChatCompletion

In [39]:
print(ai_response_openai.choices[0].message.content)

Hey, can you shoot me that report by tomorrow? Thanks!


In [40]:
#Explain concept of context length here - Context length = input tokens + completion tokens 

### Output Parsers

In [41]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

parser = StrOutputParser()

*StrOutputParser* is a runnable object.

In [42]:
result = llm.invoke(messages)

parser.invoke(result)

'Hey, could you send me the report by tomorrow? Thanks!'

In [43]:
result.content ## NOT DOING THIS ANYMORE, USING THE PARSER INSTEAD

'Hey, could you send me the report by tomorrow? Thanks!'

In [44]:
result

AIMessage(content='Hey, could you send me the report by tomorrow? Thanks!', additional_kwargs={'reasoning_content': 'User wants informal tone, but also corrected text. Original: "hey can you send me that report by tomorrow thx". Need informal but still corrected. Perhaps: "Hey, could you send me the report by tomorrow? Thanks!" That\'s informal. Provide that.'}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 158, 'total_tokens': 234, 'completion_time': 0.160264239, 'prompt_time': 0.006767939, 'queue_time': 0.354258587, 'total_time': 0.167032178, 'completion_tokens_details': {'reasoning_tokens': 54}}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_1d982b31b2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--7fc314c0-0db6-4aa6-952b-36bccc6aca87-0', usage_metadata={'input_tokens': 158, 'output_tokens': 76, 'total_tokens': 234})

In [45]:
messages = [
    SystemMessage(content="""You are an expert in writing analysis. You will receive a message from a user, and your job is to evaluate the text based on the following attributes:
1. clarity: Is the message clear, or unclear?
2. grammar_quality: Are there any grammatical issues? Possible values: correct, minor issues, major issues.
3. tone: Analyze whether the tone is neutral, formal, or informal.
4. suggestions: Offer brief improvement suggestions for clarity, grammar, or tone.

Return a structured JSON object with these four attributes. Wrap the JSON between ```json tags"""),
    HumanMessage(content="Hey, could you please send me that report by tomorrow? Thank you.")
]
response = llm.invoke(messages)

In [46]:
response

AIMessage(content='```json\n{\n  "clarity": "clear",\n  "grammar_quality": "correct",\n  "tone": "informal",\n  "suggestions": "Consider using a more formal greeting (e.g., \\"Hello\\") or removing \\"Hey\\" for a professional tone. The request itself is clear and polite."\n}\n```', additional_kwargs={'reasoning_content': 'We need to output JSON with fields: clarity, grammar_quality, tone, suggestions. Evaluate.\n\nMessage: "Hey, could you please send me that report by tomorrow? Thank you."\n\nClarity: clear, request is clear. So clarity: clear.\n\nGrammar: "Hey, could you please send me that report by tomorrow? Thank you." Grammar is correct. So grammar_quality: correct.\n\nTone: informal greeting "Hey", but also polite. So tone: informal (maybe neutral? But "Hey" is informal). So tone: informal.\n\nSuggestions: maybe suggest a more formal tone if needed, or remove "Hey". Could suggest "Please send me the report by tomorrow." But keep polite. So suggestions: maybe replace "Hey" with "

In [47]:
print(response.content)

```json
{
  "clarity": "clear",
  "grammar_quality": "correct",
  "tone": "informal",
  "suggestions": "Consider using a more formal greeting (e.g., \"Hello\") or removing \"Hey\" for a professional tone. The request itself is clear and polite."
}
```


In [48]:
json_response = JsonOutputParser().invoke(response.content)
json_response

{'clarity': 'clear',
 'grammar_quality': 'correct',
 'tone': 'informal',
 'suggestions': 'Consider using a more formal greeting (e.g., "Hello") or removing "Hey" for a professional tone. The request itself is clear and polite.'}

In [49]:
var1 = '{"clarity": "unclear"}'
print(var1)

{"clarity": "unclear"}


In [50]:
var1

'{"clarity": "unclear"}'

In [51]:
# JsonOutputParser().invoke(var1)

In [52]:
type(json_response)

dict

In [53]:
type(var1)

str

In [54]:
print(response.content)

```json
{
  "clarity": "clear",
  "grammar_quality": "correct",
  "tone": "informal",
  "suggestions": "Consider using a more formal greeting (e.g., \"Hello\") or removing \"Hey\" for a professional tone. The request itself is clear and polite."
}
```


In [55]:
json_response['clarity']

'clear'

### Chain (LCEL)

In [56]:
chain = llm | JsonOutputParser()

chain.invoke(messages)

{'clarity': 'clear',
 'grammar_quality': 'correct',
 'tone': 'informal',
 'suggestions': 'If a more formal tone is desired, replace "Hey" with "Hello" or "Dear [Name]" and consider adding a polite closing.'}

### PromptTemplate

In [57]:
# my_str = "Hello world, this is the tone - {tone}"

In [58]:
# my_str.format(tone = "Happy")

In [59]:
# tone = "Happy"
# my_str = f"Hello world, this is the tone - {tone}"
# print(my_str)

In [60]:
my_str = """Detect and correct all spelling, grammatical, and punctuation errors in the provided text.
Ensure proper sentence structure, clarity, and readability.

Tone Adjustment: {tone}

Communication Style: {communication_style}

Retain the core message of the original text while making the necessary corrections and tone adjustments.
Suggest appropriate phrasing and formatting based on the tone and communication style."""

In [61]:
print(my_str.format(tone="Happy", communication_style = "Formal"))

Detect and correct all spelling, grammatical, and punctuation errors in the provided text.
Ensure proper sentence structure, clarity, and readability.

Tone Adjustment: Happy

Communication Style: Formal

Retain the core message of the original text while making the necessary corrections and tone adjustments.
Suggest appropriate phrasing and formatting based on the tone and communication style.


In [62]:
from langchain_core.prompts import ChatPromptTemplate
system_message_template = """Detect and correct all spelling, grammatical, and punctuation errors in the provided text.
Ensure proper sentence structure, clarity, and readability.

Tone Adjustment: {tone}

Communication Style: {communication_style}

Retain the core message of the original text while making the necessary corrections and tone adjustments.
Suggest appropriate phrasing and formatting based on the tone and communication style."""

#Defining a ChatPromptTemplate
template = ChatPromptTemplate([
    ("system", system_message_template),
    ("human", "{user_input}"),
])

In [63]:
template.input_variables

['communication_style', 'tone', 'user_input']

In [64]:
#How does this work without LCEL = LangChain Expression LangChain

In [65]:
tone = 'Rewrite the message in a professional, polite, and structured manner. \
Suitable for business emails, official reports, or any context requiring formality and respect.'

communication_style = 'Messages should be clear, structured, and formal or neutral depending on the context. \
Introductions, conclusions, and appropriate sign-offs should be added if missing.'

user_input = """Can u send me the data by eod pls?"""

In [66]:
#First step
formatted_template = template.invoke({"communication_style": communication_style,
    "tone":tone,
    "user_input":user_input
})
formatted_template

ChatPromptValue(messages=[SystemMessage(content='Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\nEnsure proper sentence structure, clarity, and readability.\n\nTone Adjustment: Rewrite the message in a professional, polite, and structured manner. Suitable for business emails, official reports, or any context requiring formality and respect.\n\nCommunication Style: Messages should be clear, structured, and formal or neutral depending on the context. Introductions, conclusions, and appropriate sign-offs should be added if missing.\n\nRetain the core message of the original text while making the necessary corrections and tone adjustments.\nSuggest appropriate phrasing and formatting based on the tone and communication style.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Can u send me the data by eod pls?', additional_kwargs={}, response_metadata={})])

In [67]:
type(formatted_template)

langchain_core.prompt_values.ChatPromptValue

In [68]:
#Second step
response = llm.invoke(formatted_template)

In [69]:
type(response)

langchain_core.messages.ai.AIMessage

In [70]:
response.content

'**Subject:** Request for Data by End of Day  \n\nDear\u202f[Recipient’s Name],\n\nI hope you are well. Could you please send me the requested data by the end of the day? Your prompt assistance would be greatly appreciated.\n\nThank you for your attention to this matter.\n\nBest regards,  \n[Your Name]  \n[Your Position]  \n[Your Company]  \n[Contact Information]'

In [71]:
#Third step
final_parsed_result = StrOutputParser().invoke(response)
final_parsed_result

'**Subject:** Request for Data by End of Day  \n\nDear\u202f[Recipient’s Name],\n\nI hope you are well. Could you please send me the requested data by the end of the day? Your prompt assistance would be greatly appreciated.\n\nThank you for your attention to this matter.\n\nBest regards,  \n[Your Name]  \n[Your Position]  \n[Your Company]  \n[Contact Information]'

In [72]:
# #With LCEL
# proof_read_chain = template | llm ##RunnableSequence
# final_response = proof_read_chain.invoke(
#     {"communication_style": communication_style,
#     "tone":tone,
#     "user_input":user_input
# })
# final_response
# type(final_response)

In [73]:
#With LCEL
proof_read_chain = template | llm | StrOutputParser()
proof_read_chain

ChatPromptTemplate(input_variables=['communication_style', 'tone', 'user_input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['communication_style', 'tone'], input_types={}, partial_variables={}, template='Detect and correct all spelling, grammatical, and punctuation errors in the provided text.\nEnsure proper sentence structure, clarity, and readability.\n\nTone Adjustment: {tone}\n\nCommunication Style: {communication_style}\n\nRetain the core message of the original text while making the necessary corrections and tone adjustments.\nSuggest appropriate phrasing and formatting based on the tone and communication style.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['user_input'], input_types={}, partial_variables={}, template='{user_input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x168daa510>, async_client=<groq.

In [74]:
type(proof_read_chain)

langchain_core.runnables.base.RunnableSequence

In [75]:
final_response = proof_read_chain.invoke(
    {"communication_style": communication_style,
    "tone":tone,
    "user_input":user_input
})
final_response

'Subject: Request for Data Submission by End of Day\n\nDear [Recipient’s Name],\n\nCould you please send me the requested data by the end of the business day today? Your prompt assistance is greatly appreciated.\n\nThank you for your cooperation.\n\nBest regards,  \n[Your Name]  \n[Your Position]  \n[Your Company]  \n[Contact Information]'

In [76]:
# proof_read_chain.input_schema.model_json_schema()

In [77]:
tone_map = {
    "Formal": "Rewrite the message in a professional, polite, and structured manner. Suitable for business emails, official reports, or any context requiring formality and respect.",
    "Informal": "Rewrite in a casual, friendly, and conversational style. Appropriate for personal communications, friendly chats, or informal emails.",
    "Neutral": "Rewrite in a balanced tone that is neither overly formal nor too casual. Suitable for most general communications where a middle-ground tone is required."
}
communication_style_map = {
    "Email": "Messages should be clear, structured, and formal or neutral depending on the context. Introductions, conclusions, and appropriate sign-offs should be added if missing.",
    "General": "This covers most forms of communication and will aim for clarity and coherence. The tone can vary as per the user's choice.",
    "Instant Messaging": "Focus on brevity, clarity, and informality, using conversational phrasing suitable for quick back-and-forth exchanges.",
    "Business Instant Messaging": "Maintain a professional but conversational tone. Messages should be concise and efficient, avoiding unnecessary formalities but keeping the language respectful."
}

In [74]:
tone = "Formal" # Formal, Informal, Neutral
communication_style = "Email" # Email, General, Instant Messaging, Business Instant Messaging
user_input = """Can u send me the data by eod pls?"""

chain_output = proof_read_chain.invoke(dict(
    tone=tone_map[tone],
    communication_style = communication_style_map[communication_style],
    user_input = user_input
))
print(chain_output)

Here's a rewritten version of the message in a professional and polite tone:

Dear [Recipient's Name],

I would appreciate it if you could provide me with the required data by the end of the day today. Please let me know if there are any issues or concerns that may prevent timely delivery.

Thank you for your prompt attention to this matter.

Best regards,
[Your Name]

Alternatively, if you'd like to convey the request in a more concise manner, you could use:

Dear [Recipient's Name],

Could you please provide the required data by the end of the day today?

Thank you for your assistance.

Best regards,
[Your Name]

This revised message maintains a professional tone while conveying the original request in a clear and structured manner.
